# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. All references to dataset elements will use their `@id` to ensure clarity and traceability.

### Dataset Source
The dataset Croissant schema is published at:
- https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and inspect the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset overview
print(f"Dataset: {getattr(metadata, 'name', '')}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Version: {getattr(metadata, 'version', '')}")
print(f"Description: {getattr(metadata, 'description', '')}\n")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"License: {getattr(metadata, 'license', '')}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

We'll print the overview of record sets and their fields in the dataset for reference. All identifiers used point to `@id` values as required by the Croissant standard.

In [ ]:
# List all record sets and their fields using their @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']}")
        print(f"  Name:   {rs.get('name', '')}")
        print("  Fields:")
        for field in rs.get('field', []):
            # field can be dict or str (@id)
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"    - {field_id}")
        print('-' * 40)

# For demonstration, print a preview of records for the first record set (if any)
if record_sets:
    first_rs_id = record_sets[0]['@id']
    print(f"\nSample record from RecordSet @id={first_rs_id}:")
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. You must use the record set and field `@id` values as listed above.

In [ ]:
dataframes = {}
rs_ids = [rs['@id'] for rs in record_sets]
# Extract all records from each record set, indexed by @id
for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

if not dataframes:
    print("No records loaded into DataFrames.")
else:
    print("Extracted DataFrames for record sets:")
    for rs_id in dataframes:
        print(f"- RecordSet @id: {rs_id}, columns: {list(dataframes[rs_id].columns)}")

    # Display first record set's columns and preview
    primary_rs_id = rs_ids[0]
    print(f"\nColumns in primary RecordSet (@id={primary_rs_id}):")
    print(dataframes[primary_rs_id].columns.tolist())
    dataframes[primary_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We perform basic EDA such as filtering, normalization, and grouping by field using `@id` references. The specific numeric and grouping fields must be selected by their identifiers from above.

In [ ]:
# For demonstration, select numeric and grouping fields from the main record set
record_set_id = rs_ids[0]
df = dataframes.get(record_set_id, pd.DataFrame())

# Auto-identify a numeric field (@id) if possible
numeric_field_id = None
if not df.empty:
    # Attempt to pick the first float/integer column, or prompt the user
    for col in df.columns:
        # Try to infer if column is numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if numeric_field_id is None:
    print("No numeric field found in the primary record set.")
# Auto-identify a categorical/grouping field
group_field_id = None
if not df.empty and numeric_field_id:
    for col in df.columns:
        if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
            group_field_id = col
            break

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("Skipping EDA: could not detect a numeric field.")

## 5. Visualization
We can visualize the distribution of the selected numeric field, or relationships between fields where available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field if identified
if not df.empty and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field is available, make boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
In this notebook, we leveraged the `mlcroissant` library to load, inspect, and analyze the FAIR^2 dataset, using only Croissant `@id` references for robust and unambiguous access. We previewed record sets, explored numeric and grouping fields, visualized key features, and demonstrated best-practice data handling. For in-depth domain analysis, further field selection and targeted data transformations are recommended.